In [5]:
# convert_voc_to_yolo.py

import os
import xml.etree.ElementTree as ET

annotations_dir = "data/raw/annotations"
images_dir = "data/raw/images"

labels_dir = "data/processed/labels"
os.makedirs(labels_dir, exist_ok=True)

classes = ["helmet", "no_helmet"]  # adjust based on dataset


def convert_box(size, box):
    dw = 1. / size[0]
    dh = 1. / size[1]

    x = (box[0] + box[1]) / 2.0
    y = (box[2] + box[3]) / 2.0

    w = box[1] - box[0]
    h = box[3] - box[2]

    x = x * dw
    w = w * dw
    y = y * dh
    h = h * dh

    return (x, y, w, h)


for file in os.listdir(annotations_dir):

    if not file.endswith(".xml"):
        continue

    tree = ET.parse(os.path.join(annotations_dir, file))
    root = tree.getroot()

    size = root.find("size")

    w = int(size.find("width").text)
    h = int(size.find("height").text)

    filename = file.replace(".xml", ".txt")

    out_file = open(os.path.join(labels_dir, filename), "w")

    for obj in root.iter("object"):

        cls = obj.find("name").text

        if cls not in classes:
            continue

        cls_id = classes.index(cls)

        xmlbox = obj.find("bndbox")

        b = (
            float(xmlbox.find("xmin").text),
            float(xmlbox.find("xmax").text),
            float(xmlbox.find("ymin").text),
            float(xmlbox.find("ymax").text)
        )

        bb = convert_box((w, h), b)

        out_file.write(
            str(cls_id) + " " + " ".join([str(a) for a in bb]) + "\n"
        )

    out_file.close()

print("Conversion Completed")

Conversion Completed


In [6]:
import os
import shutil
from sklearn.model_selection import train_test_split

# Paths
image_dir = "data/processed/images"
label_dir = "data/processed/labels"

# Get all images
images = [f for f in os.listdir(image_dir) if f.endswith(".jpg") or f.endswith(".png")]

# Split dataset
train, test = train_test_split(images, test_size=0.1, random_state=42)
train, val = train_test_split(train, test_size=0.2, random_state=42)

print("Train:", len(train))
print("Validation:", len(val))
print("Test:", len(test))


# Create folders
for split in ["train", "val", "test"]:
    
    os.makedirs(f"{image_dir}/{split}", exist_ok=True)
    os.makedirs(f"{label_dir}/{split}", exist_ok=True)


def move_files(file_list, split):

    for file in file_list:

        image_src = os.path.join(image_dir, file)
        label_src = os.path.join(label_dir, file.replace(".jpg", ".txt").replace(".png",".txt"))

        image_dst = os.path.join(image_dir, split, file)
        label_dst = os.path.join(label_dir, split, file.replace(".jpg",".txt").replace(".png",".txt"))

        if os.path.exists(image_src):
            shutil.move(image_src, image_dst)

        if os.path.exists(label_src):
            shutil.move(label_src, label_dst)


# Move files
move_files(train, "train")
move_files(val, "val")
move_files(test, "test")

print("Dataset successfully split!")

Train: 549
Validation: 138
Test: 77
Dataset successfully split!


In [7]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

model.train(
    data="data/dataset.yaml",
    epochs=1
)

ModuleNotFoundError: No module named 'ultralytics'